# Task 1.1 — Dataset Exploration & Cleaning

Goal: characterise the dataset, surface any feature that could leak the source class, and design a deterministic cleaning pipeline that downstream tasks can rely on. Outputs feed `solution/clean.py` and report §1.1.

Dataset schema (per PDF):
- labeled splits (`train`, `calibration`, `calibration_augmented`, `validation`, `validation_augmented`): `image: binary, source_class: int8`
- predict split (`predict`): `row_id: int32, image: binary`

Labels: 0=real; 1..5 collapse to 1=ai_generated.

In [ ]:
import sys
import io
from pathlib import Path

ROOT = Path.cwd().parent
SOLUTION = ROOT / "solution"
sys.path.insert(0, str(SOLUTION))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from PIL import Image

from _lib import seed
from _lib.data import binarize_label

seed.set_deterministic(0)

DATA = SOLUTION / "data"
print("data root:", DATA, "exists:", DATA.exists())

CLASS_NAMES = ["real", "SD2.1", "SDXL", "SD3", "DALL-E3", "Midjourney"]

## 1. Schema sanity

Confirm column names, row counts, and that one image decodes.

In [ ]:
splits = ["train", "calibration", "calibration_augmented",
          "validation", "validation_augmented", "predict"]

for s in splits:
    files = sorted((DATA / s).glob("*.parquet"))
    pf = pq.ParquetFile(files[0])
    print(f"{s:25s} files={len(files)}  rows/file0={pf.metadata.num_rows}  cols={pf.schema_arrow.names}")

# decode one row from train/0
pf = pq.ParquetFile(sorted((DATA / "train").glob("*.parquet"))[0])
batch = next(pf.iter_batches(batch_size=1, columns=["image", "source_class"]))
im = Image.open(io.BytesIO(batch["image"][0].as_py()))
print("sample:", im.size, im.mode, im.format, "class=", batch["source_class"][0].as_py())

## 2. Class distribution

Count source classes per split (memory-friendly: only read `source_class`). Then collapse to binary.

In [ ]:
def count_classes(split):
    counts = pd.Series(dtype=int)
    for f in sorted((DATA / split).glob("*.parquet")):
        col = pq.read_table(f, columns=["source_class"]).to_pandas()["source_class"]
        counts = counts.add(col.value_counts(), fill_value=0)
    return counts.sort_index().astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, split in zip(axes, ["train", "calibration", "validation"]):
    c = count_classes(split)
    ax.bar(range(len(c)), c.values)
    ax.set_xticks(range(len(c)))
    ax.set_xticklabels([CLASS_NAMES[i] for i in c.index], rotation=30)
    ax.set_title(f"{split} (n={c.sum()})")
plt.tight_layout()
plt.show()

for split in ["train", "calibration", "validation"]:
    c = count_classes(split)
    real, ai = int(c.get(0, 0)), int(c.drop(0, errors="ignore").sum())
    print(f"{split:12s} real={real}  ai={ai}  ai_share={ai/(real+ai):.2%}")

## 3. Image size & format

Decode a random subset (full streaming would be slow). Look for size/format patterns that correlate with class — a leak we should neutralise during cleaning.

In [ ]:
def sample_rows(split, n=1500, seed_=0):
    """Random rows from a split. Returns list of (label, bytes)."""
    rng = np.random.default_rng(seed_)
    files = sorted((DATA / split).glob("*.parquet"))
    out = []
    per_file = max(1, n // len(files) + 1)
    for f in files:
        df = pq.read_table(f).to_pandas()
        if len(df) == 0:
            continue
        take = min(per_file, len(df))
        idx = rng.choice(len(df), size=take, replace=False)
        for i in idx:
            row = df.iloc[i]
            label = int(row["source_class"]) if "source_class" in df.columns else -1
            out.append((label, row["image"]))
    rng.shuffle(out)
    return out[:n]

recs = []
for label, buf in sample_rows("train", n=2000, seed_=1):
    try:
        with Image.open(io.BytesIO(buf)) as im:
            recs.append({"class": label, "w": im.width, "h": im.height,
                         "mode": im.mode, "format": im.format, "bytes": len(buf)})
    except Exception as e:
        recs.append({"class": label, "error": str(e)[:80]})

size_df = pd.DataFrame(recs)
print("modes:\n", size_df["mode"].value_counts(dropna=False), "\n")
print("formats:\n", size_df["format"].value_counts(dropna=False), "\n")
print("per-class size stats:\n", size_df.groupby("class")[["w", "h", "bytes"]]
      .agg(["median", "min", "max"]).round(0))

fig, ax = plt.subplots(figsize=(6, 5))
ax.hist2d(size_df["w"].dropna(), size_df["h"].dropna(), bins=30, cmin=1)
ax.set_xlabel("width"); ax.set_ylabel("height"); ax.set_title("image dimensions (train sample)")
plt.show()

## 4. Per-class descriptive stats

Mean/std RGB and file-size bytes per source class. If any one stat clearly separates real vs AI, the model could learn that shortcut — worth flagging in the report.

In [ ]:
rows = []
for label, buf in sample_rows("train", n=600, seed_=2):
    try:
        a = np.asarray(Image.open(io.BytesIO(buf)).convert("RGB"), dtype=np.float32) / 255.0
        rows.append({
            "class": label,
            "mean_r": a[..., 0].mean(), "mean_g": a[..., 1].mean(), "mean_b": a[..., 2].mean(),
            "std": float(a.std()), "bytes": len(buf),
        })
    except Exception:
        pass

stat_df = pd.DataFrame(rows).groupby("class").mean().round(3)
stat_df.index = [CLASS_NAMES[i] for i in stat_df.index]
stat_df

## 5. Sample grid

One row per source class — visual sanity check.

In [ ]:
buckets = {c: [] for c in range(6)}
for label, buf in sample_rows("train", n=4000, seed_=3):
    if 0 <= label < 6 and len(buckets[label]) < 6:
        buckets[label].append(buf)
    if all(len(v) == 6 for v in buckets.values()):
        break

fig, axes = plt.subplots(6, 6, figsize=(12, 12))
for c in range(6):
    for j in range(6):
        ax = axes[c, j]
        ax.set_xticks([]); ax.set_yticks([])
        if j < len(buckets[c]):
            ax.imshow(Image.open(io.BytesIO(buckets[c][j])).convert("RGB"))
        if j == 0:
            ax.set_ylabel(CLASS_NAMES[c], fontsize=10)
plt.tight_layout()
plt.show()

## 6. Cleaning pipeline draft

Decisions, justified by what cells 2–4 surfaced:

- **Resize-shorter-edge then center-crop to 224×224.** Matches `train_time_reference.py` and the Appendix B CNN's expected input. Preserves aspect ratio (avoids the leak where stretching reveals class-specific original dimensions).
- **Convert all images to RGB.** Drops grayscale/RGBA divergence — mode itself shouldn't be a feature.
- **Drop unreadable rows** (corrupt bytes, exotic formats). Cleaner training signal; quantify the drop rate.
- **No deduplication** for now — will revisit if we see suspicious near-duplicates between train and validation.
- **Don't strip JPEG metadata-derived features yet** — the classical baseline may want file-size as a feature; keep it as an auxiliary column.

In [ ]:
IMG_SIZE = 224  # Appendix B CNN input; reference timing also uses this

def clean_image(buf):
    """bytes -> (224, 224, 3) float32 in [0, 1], or None if unreadable.

    Resize-shorter-edge + center-crop preserves aspect; we don't want stretching
    to encode original dimensions.
    """
    try:
        im = Image.open(io.BytesIO(buf)).convert("RGB")
    except Exception:
        return None
    w, h = im.size
    s = IMG_SIZE / min(w, h)
    im = im.resize((int(round(w * s)), int(round(h * s))), Image.BILINEAR)
    nw, nh = im.size
    left = (nw - IMG_SIZE) // 2
    top = (nh - IMG_SIZE) // 2
    im = im.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))
    return np.asarray(im, dtype=np.float32) / 255.0

buf = sample_rows("train", n=1, seed_=4)[0][1]
out = clean_image(buf)
print("shape:", out.shape, "dtype:", out.dtype, "range:", (float(out.min()), float(out.max())))

## 7. Validate cleaning on a 500-row subset

Smoke test: every row produces a 224×224×3 float32 in [0, 1], or is reported as dropped.

In [ ]:
n_total, n_dropped = 0, 0
shape_ok, range_ok = True, True
for _, buf in sample_rows("train", n=500, seed_=5):
    n_total += 1
    out = clean_image(buf)
    if out is None:
        n_dropped += 1
        continue
    if out.shape != (IMG_SIZE, IMG_SIZE, 3) or out.dtype != np.float32:
        shape_ok = False
    if out.min() < 0.0 or out.max() > 1.0:
        range_ok = False

assert shape_ok, "shape/dtype check failed"
assert range_ok, "value range check failed"
print(f"smoke test: {n_total} rows, {n_dropped} dropped, all kept rows are 224x224x3 float32 in [0, 1]")

## 8. Port targets — hand-off to `solution/`

When this notebook converges, port:

- `_lib/io.py::read_parquet_split(split_dir)` ← streaming version of `sample_rows` that yields all rows (no random sampling, no eager `to_pandas`)
- `_lib/io.py::decode_image(buf)` ← `Image.open(BytesIO(buf)).convert("RGB")` returning PIL or array; expose `IMG_SIZE` constant here
- `_lib/data.py::LabeledImageDataset.__iter__` ← combine the two helpers + `binarize_label(source_class)`
- `clean.py::explore_class_distribution` ← cell 2
- `clean.py::explore_image_sizes` ← cell 3
- `clean.py::explore_descriptive_stats` ← cell 4
- `clean.py::clean_pipeline` ← `clean_image()`
- `clean.py::save_cleaned_dataset` ← write cleaned tensors/parquet under `artifacts/clean/` (deferred until prepare.py is needed)

Figures from cells 2, 3, 5 land in report §1.1.